# LeetCode 1055 — Shortest Way to Form String bằng Python

Link đề: https://leetcode.com/problems/shortest-way-to-form-string/ · Độ khó: **Medium** · Premium (cần tài khoản LeetCode Premium để xem đề) · Chủ đề: Greedy (tham lam), Two Pointers (hai con trỏ), String (chuỗi), Binary Search (tìm kiếm nhị phân)

Skeleton LeetCode:

```python
class Solution:
    def shortestWay(self, source: str, target: str) -> int:
        ...
```

Ghi chú: một **dãy con** (subsequence) của chuỗi là chuỗi thu được bằng cách xóa một số ký tự (có thể không xóa) mà không đổi thứ tự các ký tự còn lại. Cho hai chuỗi `source` và `target`, trả về **số dãy con ít nhất** của `source` mà khi **nối** lại bằng đúng `target`; nếu không thể thì trả về `-1`. Ý tưởng chính: **tham lam**: mỗi dãy con lấy **dài nhất có thể**, tức quét `source` từ trái sang phải một lượt và khớp càng nhiều ký tự liên tiếp của `target` càng tốt; hết `source` thì bắt đầu một dãy con mới.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu **dãy con** khác **chuỗi con** (substring) thế nào, và bài toán chia `target` thành ít đoạn nhất, mỗi đoạn là một dãy con của `source`.
2. Nắm các công cụ Python cần thiết: hai con trỏ trên chuỗi, `set` (kiểm tra ký tự thiếu), `str.find`, `bisect_left`, mảng "vị trí xuất hiện kế tiếp".
3. Hiểu chứng minh tham lam bằng **lập luận trao đổi / đơn điệu**: bắt đầu càng xa trong `target` thì tiến được càng xa.
4. Đi qua 5 cách: vét cạn mọi cách cắt `target` (brute force, dùng làm oracle), quy hoạch động theo tiền tố, hai con trỏ tham lam (cách nộp chính), bảng "ký tự kế tiếp" và tìm kiếm nhị phân trên danh sách vị trí.
5. Tránh các lỗi kinh điển: quên kiểm tra ký tự không có trong `source` (vòng lặp vô hạn), nhầm dãy con với chuỗi con, đếm ký tự rồi chia, lệch một khi dùng `bisect`, và đảo vai trò `source`/`target`.

## 2. Đề bài & ví dụ

Một **dãy con** (subsequence) của một chuỗi là chuỗi mới tạo ra từ chuỗi gốc bằng cách xóa một số ký tự (có thể không xóa ký tự nào) mà **không thay đổi vị trí tương đối** của các ký tự còn lại. Ví dụ `"ace"` là dãy con của `"abcde"` còn `"aec"` thì không.

Cho hai chuỗi `source` và `target`, trả về **số lượng dãy con ít nhất của `source`** sao cho phép **nối** chúng bằng đúng `target`. Nếu không thể, trả về `-1`.

**Ví dụ 1:** `source = "abc"`, `target = "abcbc"` -> `2`. `"abcbc"` = `"abc"` + `"bc"`, cả hai là dãy con của `"abc"`.

**Ví dụ 2:** `source = "abc"`, `target = "acdbc"` -> `-1`, vì `target` có ký tự `d` không có trong `source`.

**Ví dụ 3:** `source = "xyz"`, `target = "xzyxz"` -> `3`: `"xz"` + `"y"` + `"xz"`.

**Ràng buộc:**

- `1 <= source.length, target.length <= 1000`
- `source` và `target` gồm các chữ cái tiếng Anh viết thường.

In [1]:
from typing import List, Optional
from itertools import product
from bisect import bisect_left, bisect_right
from collections import defaultdict
import random, time

examples = [
    ("abc", "abcbc", 2),
    ("abc", "acdbc", -1),
    ("xyz", "xzyxz", 3),
]
for source, target, expected in examples:
    print(f"source={source!r} target={target!r} -> kỳ vọng {expected}")

source='abc' target='abcbc' -> kỳ vọng 2
source='abc' target='acdbc' -> kỳ vọng -1
source='xyz' target='xzyxz' -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1. Dãy con và chuỗi con

Chuỗi con (substring) phải **liên tiếp**; dãy con (subsequence) chỉ cần giữ thứ tự. Kiểm tra `p` là dãy con của `s` bằng hai con trỏ, hoặc bằng thủ thuật `iter`: `all(c in it for c in p)` với `it = iter(s)` (mỗi `c in it` tiêu thụ iterator tới khi gặp `c`).

In [2]:
def is_subsequence(p: str, s: str) -> bool:
    it = iter(s)
    return all(c in it for c in p)

print(is_subsequence("ace", "abcde"), is_subsequence("aec", "abcde"))
print("ace" in "abcde")                      # toán tử in trên chuỗi kiểm tra CHUỖI CON, không phải dãy con

True False
False


### 3.2. Hai con trỏ

Một con trỏ `i` chạy trên `source`, một con trỏ `j` chạy trên `target`. Khi `source[i] == target[j]` thì tiến `j`. `i` luôn tiến. Hết `source` là kết thúc **một dãy con**.

In [3]:
source, target = "abc", "abcbc"
i = j = 0
while i < len(source) and j < len(target):
    print(f"i={i} source[i]={source[i]!r} j={j} target[j]={target[j]!r}", "khớp" if source[i] == target[j] else "")
    if source[i] == target[j]:
        j += 1
    i += 1
print("dãy con đầu tiên khớp được:", target[:j])

i=0 source[i]='a' j=0 target[j]='a' khớp
i=1 source[i]='b' j=1 target[j]='b' khớp
i=2 source[i]='c' j=2 target[j]='c' khớp
dãy con đầu tiên khớp được: abc


### 3.3. Danh sách vị trí và `bisect_left`

Với mỗi ký tự lưu danh sách các vị trí xuất hiện trong `source` (đã tăng dần). `bisect_left(lst, pos)` trả về chỉ số phần tử đầu tiên `>= pos`: vị trí **xuất hiện kế tiếp** của ký tự đó tại hoặc sau `pos`, trong `O(log)`.

In [4]:
source = "abcab"
idx = defaultdict(list)
for k, ch in enumerate(source):
    idx[ch].append(k)
print(dict(idx))
lst = idx["a"]
print("a kế tiếp từ vị trí 1:", lst[bisect_left(lst, 1)], "| từ vị trí 0:", lst[bisect_left(lst, 0)], "| từ vị trí 4:", bisect_left(lst, 4), "(= len(lst): hết)")

{'a': [0, 3], 'b': [1, 4], 'c': [2]}
a kế tiếp từ vị trí 1: 3 | từ vị trí 0: 0 | từ vị trí 4: 2 (= len(lst): hết)


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 3: `source = "xyz"`, `target = "xzyxz"`.

- Dãy con thứ nhất: quét `source`: `x` khớp `x`; `y` không khớp `z`; `z` khớp `z`. Được `"xz"`. Hết `source`.
- Dãy con thứ hai: `x` không khớp `y`; `y` khớp `y`; `z` không khớp `x`. Được `"y"`.
- Dãy con thứ ba: `x` khớp `x`; `y` không khớp `z`; `z` khớp `z`. Được `"xz"`.
- Tổng `3` ✓.

Ví dụ 2: `target` có `d`, `source` không có `d` ⇒ không bao giờ khớp được ⇒ `-1`.

Câu hỏi: **vì sao lấy dãy con dài nhất mỗi lần là tối ưu?** Đoạn nào cũng phải bắt đầu tại ký tự tiếp theo của `target`; càng đi được xa thì phần còn lại càng ngắn và (như sẽ chứng minh) không bao giờ tệ hơn.

## 5. Cách 1 — Brute Force: thử mọi cách cắt `target`

**Ý tưởng.** `target` có `n` ký tự nên có `n - 1` chỗ có thể cắt (giữa hai ký tự liền nhau). Mỗi tập chỗ cắt cho một cách chia thành các đoạn; cách chia hợp lệ nếu **mọi đoạn là dãy con của `source`**. Lấy số đoạn nhỏ nhất. Đúng định nghĩa đề nên đây là **oracle độc lập** (không dùng tham lam, không dùng quy hoạch động).

In [5]:
def shortest_cuts(source: str, target: str) -> int:
    n = len(target)
    best = -1
    for mask in range(1 << (n - 1)):                       # bit k = 1: cắt giữa target[k] và target[k + 1]
        pieces, start = [], 0
        for k in range(n - 1):
            if mask >> k & 1:
                pieces.append(target[start:k + 1])
                start = k + 1
        pieces.append(target[start:])
        if all(is_subsequence(p, source) for p in pieces):
            if best == -1 or len(pieces) < best:
                best = len(pieces)
    return best

for source, target, expected in examples:
    got = shortest_cuts(source, target)
    print(f"cắt mọi cách({source!r}, {target!r}) = {got}", "OK" if got == expected else "SAI")
    assert got == expected

cắt mọi cách('abc', 'abcbc') = 2 OK
cắt mọi cách('abc', 'acdbc') = -1 OK
cắt mọi cách('xyz', 'xzyxz') = 3 OK


**Độ phức tạp và nút thắt.** Có `2^(n - 1)` cách cắt, mỗi cách tốn `O(n * m)` để kiểm tra ⇒ mũ. Đo thử khi `n` tăng:

In [6]:
rng = random.Random(1055)
for n in (8, 10, 12, 14, 16):
    target = "".join(rng.choice("ab") for _ in range(n))
    t0 = time.perf_counter()
    shortest_cuts("ab", target)
    dt = time.perf_counter() - t0
    print(f"n={n:2}: {dt * 1000:9.1f} ms")

n= 8:       0.2 ms
n=10:       1.1 ms
n=12:       7.4 ms
n=14:      30.9 ms


n=16:     140.4 ms


Nút thắt: ta xét lại rất nhiều lần cùng một **tiền tố** của `target`. Chỉ cần biết "số đoạn ít nhất để tạo ra tiền tố độ dài `j`".

## 6. Ý tưởng thuật toán / bước đột phá

**Bước 1 (quy hoạch động theo tiền tố).** Gọi `dp[j]` là số dãy con ít nhất để tạo `target[:j]`. Đoạn cuối là `target[i:j]` phải là dãy con của `source`: `dp[j] = min(dp[i] + 1)` theo các `i < j` thỏa điều kiện. Polynomial nhưng nặng.

**Bước 2 (bước đột phá: tham lam).** Gọi `reach(p)` là chỉ số `q` lớn nhất sao cho `target[p:q]` là dãy con của `source`. Nếu `p >= p'` thì `reach(p) >= reach(p')`: vì nếu `target[p':q']` là dãy con của `source` và `q' >= p` thì `target[p:q']` (một phần cuối của nó) cũng là dãy con của `source`. Do đó đi càng xa càng tốt: sau `k` đoạn, cách tham lam luôn đã khớp được nhiều ký tự nhất có thể. Vậy **mỗi đoạn lấy dài nhất** (quét `source` một lượt, khớp càng nhiều càng tốt) là tối ưu.

**Bước 3 (khi nào `-1`).** Nếu `target` có ký tự không có trong `source`, không thể. Ngược lại mỗi lượt quét khớp được ít nhất một ký tự (ký tự cần khớp đều xuất hiện trong `source`), nên thuật toán kết thúc sau tối đa `len(target)` đoạn.

| Cách | Ý tưởng |
|---|---|
| 2 | DP theo tiền tố: `dp[j] = min(dp[i] + 1)` |
| 3 | Hai con trỏ tham lam, mỗi lượt quét `source` (nộp) |
| 4 | Bảng ký tự kế tiếp `nxt[i][c]`: mỗi ký tự `target` xử lý `O(1)` |
| 5 | Danh sách vị trí + `bisect_left`: mỗi ký tự `target` xử lý `O(log)` |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Quy hoạch động theo tiền tố

**Giải thích từng đoạn.**

- `dp[0] = 0`; `dp[j] = inf` nếu chưa tạo được.
- Với mỗi `j`, thử mọi `i < j`; nếu `dp[i]` hữu hạn và `target[i:j]` là dãy con của `source` thì `dp[j] = min(dp[j], dp[i] + 1)`.
- Đáp án `dp[n]` hoặc `-1` nếu vô hạn.

In [7]:
def shortest_dp(source: str, target: str) -> int:
    n = len(target)
    INF = float("inf")
    dp = [0] + [INF] * n
    for j in range(1, n + 1):
        for i in range(j):
            if dp[i] + 1 < dp[j] and is_subsequence(target[i:j], source):
                dp[j] = dp[i] + 1
    return -1 if dp[n] == INF else dp[n]

for source, target, expected in examples:
    got = shortest_dp(source, target)
    print(f"DP({source!r}, {target!r}) = {got}", "OK" if got == expected else "SAI")
    assert got == expected

DP('abc', 'abcbc') = 2 OK
DP('abc', 'acdbc') = -1 OK
DP('xyz', 'xzyxz') = 3 OK


**Code nộp LeetCode (Cách 2):** đúng nhưng chậm (`O(n^2 * m)`), chỉ để đối chiếu, sẽ quá thời gian với `n = m = 1000`.

In [8]:
class Solution:
    def shortestWay(self, source: str, target: str) -> int:
        n = len(target)
        INF = float("inf")
        dp = [0] + [INF] * n                           # dp[j]: ít dãy con nhất tạo target[:j]
        for j in range(1, n + 1):
            for i in range(j):
                if dp[i] + 1 < dp[j]:
                    it = iter(source)
                    if all(c in it for c in target[i:j]):    # target[i:j] là dãy con của source?
                        dp[j] = dp[i] + 1
        return -1 if dp[n] == INF else dp[n]

sol_dp = lambda s, t: Solution().shortestWay(s, t)
for source, target, expected in examples:
    assert sol_dp(source, target) == expected
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** thời gian `O(n^2 * m)`, bộ nhớ `O(n)`.

Đo để thấy vì sao không đủ nhanh:

In [9]:
rng = random.Random(7)
for n in (50, 100, 200):
    source = "".join(rng.choice("abc") for _ in range(n))
    target = "".join(rng.choice("abc") for _ in range(n))
    t0 = time.perf_counter()
    shortest_dp(source, target)
    dt = time.perf_counter() - t0
    print(f"n=m={n:3}: {dt * 1000:8.1f} ms -> ngoại suy n=m=1000: ~{dt * (1000 / n) ** 3:8.1f} s")

n=m= 50:      1.9 ms -> ngoại suy n=m=1000: ~    15.0 s
n=m=100:     11.4 ms -> ngoại suy n=m=1000: ~    11.4 s


n=m=200:     72.1 ms -> ngoại suy n=m=1000: ~     9.0 s


---

### 7.2. Cách 3 — Hai con trỏ tham lam (cách nộp chính)

**Giải thích từng đoạn.**

- Nếu `set(target) - set(source)` khác rỗng: có ký tự không thể tạo ⇒ `-1`.
- `j` là vị trí hiện tại trong `target`. Mỗi vòng ngoài là một dãy con mới: `i` quét `source` từ đầu; khi `source[i] == target[j]` thì `j += 1`.
- Mỗi vòng ngoài cộng `1` vào kết quả. Dừng khi `j == len(target)`.

In [10]:
def shortest_two_pointers(source: str, target: str) -> int:
    if set(target) - set(source):
        return -1
    pieces, j = 0, 0
    while j < len(target):
        i = 0
        while i < len(source) and j < len(target):
            if source[i] == target[j]:
                j += 1
            i += 1
        pieces += 1
    return pieces

for source, target, expected in examples:
    got = shortest_two_pointers(source, target)
    print(f"hai con trỏ({source!r}, {target!r}) = {got}", "OK" if got == expected else "SAI")
    assert got == expected

hai con trỏ('abc', 'abcbc') = 2 OK
hai con trỏ('abc', 'acdbc') = -1 OK
hai con trỏ('xyz', 'xzyxz') = 3 OK


**Code nộp LeetCode (Cách 3):**

In [11]:
class Solution:
    def shortestWay(self, source: str, target: str) -> int:
        if set(target) - set(source):                  # có ký tự không tồn tại trong source
            return -1
        m, n = len(source), len(target)
        pieces = 0                                     # số dãy con đã dùng
        j = 0                                          # vị trí cần khớp trong target
        while j < n:
            i = 0
            while i < m and j < n:                     # một lượt quét source = một dãy con
                if source[i] == target[j]:
                    j += 1
                i += 1
            pieces += 1
        return pieces

sol_two = lambda s, t: Solution().shortestWay(s, t)
for source, target, expected in examples:
    assert sol_two(source, target) == expected
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** mỗi lượt quét `O(m)` và có tối đa `n` lượt ⇒ `O(m * n)` trong trường hợp xấu (thực tế mỗi lượt quét đã khớp ít nhất một ký tự), bộ nhớ `O(1)` (không tính `set`, tối đa `26` ký tự).

---

### 7.3. Cách 4 — Bảng "ký tự kế tiếp" `nxt`

**Ý tưởng khác.** Tiền xử lý `nxt[i][c]` = chỉ số nhỏ nhất `>= i` trong `source` có ký tự `c` (hoặc `-1` nếu không có). Khi đã biết `pos` (vị trí tiếp theo được phép dùng trong `source`), ký tự `target` tiếp theo khớp ngay tại `nxt[pos][c]`, không phải quét từng ký tự; nếu không còn thì bắt đầu dãy con mới (`pos = 0`).

Bảng có `(m + 1) * 26` ô; xử lý `target` mất `O(n)`.

In [12]:
def shortest_next_table(source: str, target: str) -> int:
    m = len(source)
    nxt = [[-1] * 26 for _ in range(m + 1)]
    for i in range(m - 1, -1, -1):
        nxt[i] = nxt[i + 1][:]
        nxt[i][ord(source[i]) - 97] = i
    pieces, pos = 1, 0
    for ch in target:
        c = ord(ch) - 97
        if nxt[0][c] == -1:
            return -1
        if nxt[pos][c] == -1:                    # hết source: sang dãy con mới
            pieces += 1
            pos = 0
        pos = nxt[pos][c] + 1
    return pieces

for source, target, expected in examples:
    got = shortest_next_table(source, target)
    print(f"bảng nxt({source!r}, {target!r}) = {got}", "OK" if got == expected else "SAI")
    assert got == expected

bảng nxt('abc', 'abcbc') = 2 OK
bảng nxt('abc', 'acdbc') = -1 OK
bảng nxt('xyz', 'xzyxz') = 3 OK


**Code nộp LeetCode (Cách 4):**

In [13]:
class Solution:
    def shortestWay(self, source: str, target: str) -> int:
        m = len(source)
        nxt = [[-1] * 26 for _ in range(m + 1)]        # nxt[i][c]: chỉ số nhỏ nhất >= i của ký tự c (hoặc -1)
        for i in range(m - 1, -1, -1):
            nxt[i] = nxt[i + 1][:]
            nxt[i][ord(source[i]) - 97] = i
        pieces, pos = 1, 0                              # đang ở dãy con thứ nhất, vị trí được dùng tiếp theo trong source
        for ch in target:
            c = ord(ch) - 97
            if nxt[0][c] == -1:                         # ký tự không có trong source
                return -1
            if nxt[pos][c] == -1:                       # hết source: bắt đầu dãy con mới
                pieces += 1
                pos = 0
            pos = nxt[pos][c] + 1
        return pieces

sol_table = lambda s, t: Solution().shortestWay(s, t)
for source, target, expected in examples:
    assert sol_table(source, target) == expected
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** thời gian `O(26 * m + n)`, bộ nhớ `O(26 * m)`.

---

### 7.4. Cách 5 — Danh sách vị trí + `bisect_left`

**Ý tưởng khác.** Thay bảng `26 * m` bằng danh sách vị trí của từng ký tự (tổng độ dài `m`): với `pos` hiện tại, `bisect_left(lst, pos)` tìm lần xuất hiện kế tiếp của `ch`. Nếu vượt cuối danh sách thì bắt đầu dãy con mới và lấy phần tử đầu `lst[0]`.

In [14]:
def shortest_bisect(source: str, target: str) -> int:
    idx = defaultdict(list)
    for k, ch in enumerate(source):
        idx[ch].append(k)
    pieces, pos = 1, 0
    for ch in target:
        if ch not in idx:
            return -1
        lst = idx[ch]
        k = bisect_left(lst, pos)
        if k == len(lst):                         # không còn ch từ vị trí pos
            pieces += 1
            k = 0
        pos = lst[k] + 1
    return pieces

for source, target, expected in examples:
    got = shortest_bisect(source, target)
    print(f"bisect({source!r}, {target!r}) = {got}", "OK" if got == expected else "SAI")
    assert got == expected

bisect('abc', 'abcbc') = 2 OK
bisect('abc', 'acdbc') = -1 OK
bisect('xyz', 'xzyxz') = 3 OK


**Code nộp LeetCode (Cách 5):**

In [15]:
class Solution:
    def shortestWay(self, source: str, target: str) -> int:
        from bisect import bisect_left
        from collections import defaultdict
        idx = defaultdict(list)                         # idx[ch]: các vị trí của ch trong source (tăng dần)
        for k, ch in enumerate(source):
            idx[ch].append(k)
        pieces, pos = 1, 0
        for ch in target:
            if ch not in idx:
                return -1
            lst = idx[ch]
            k = bisect_left(lst, pos)                   # lần xuất hiện đầu tiên tại hoặc sau pos
            if k == len(lst):                           # hết source: sang dãy con mới
                pieces += 1
                k = 0
            pos = lst[k] + 1
        return pieces

sol_bisect = lambda s, t: Solution().shortestWay(s, t)
for source, target, expected in examples:
    assert sol_bisect(source, target) == expected
print("Cách 5 qua ví dụ")

Cách 5 qua ví dụ


**Độ phức tạp (Cách 5):** thời gian `O(m + n log m)`, bộ nhớ `O(m)`.

## 8. TRACE từng bước

### 8.1. Hai con trỏ cho `source = "xyz"`, `target = "xzyxz"`

In từng lượt quét và phần của `target` được khớp.

In [16]:
def trace_two_pointers(source, target):
    if set(target) - set(source):
        print("có ký tự thiếu:", sorted(set(target) - set(source)), "-> -1")
        return -1
    pieces, j = 0, 0
    while j < len(target):
        start = j
        i = 0
        print(f"--- dãy con #{pieces + 1}, bắt đầu từ target[{j}] = {target[j]!r}")
        while i < len(source) and j < len(target):
            match = source[i] == target[j]
            print(f"  i={i} source[i]={source[i]!r} | j={j} target[j]={target[j]!r} | {'khớp' if match else 'không khớp'}")
            if match:
                j += 1
            i += 1
        pieces += 1
        print(f"  => dãy con #{pieces} = {target[start:j]!r}")
    print("tổng:", pieces)
    return pieces

assert trace_two_pointers("xyz", "xzyxz") == 3
print()
assert trace_two_pointers("abc", "acdbc") == -1

--- dãy con #1, bắt đầu từ target[0] = 'x'
  i=0 source[i]='x' | j=0 target[j]='x' | khớp
  i=1 source[i]='y' | j=1 target[j]='z' | không khớp
  i=2 source[i]='z' | j=1 target[j]='z' | khớp
  => dãy con #1 = 'xz'
--- dãy con #2, bắt đầu từ target[2] = 'y'
  i=0 source[i]='x' | j=2 target[j]='y' | không khớp
  i=1 source[i]='y' | j=2 target[j]='y' | khớp
  i=2 source[i]='z' | j=3 target[j]='x' | không khớp
  => dãy con #2 = 'y'
--- dãy con #3, bắt đầu từ target[3] = 'x'
  i=0 source[i]='x' | j=3 target[j]='x' | khớp
  i=1 source[i]='y' | j=4 target[j]='z' | không khớp
  i=2 source[i]='z' | j=4 target[j]='z' | khớp
  => dãy con #3 = 'xz'
tổng: 3

có ký tự thiếu: ['d'] -> -1


### 8.2. Bảng `nxt` và tìm kiếm nhị phân cho `source = "abcab"`, `target = "ababcab"`

In bảng `nxt` (chỉ các ký tự `a, b, c`) và con trỏ `pos` mỗi bước.

In [17]:
source, target = "abcab", "ababcab"
m = len(source)
nxt = [[-1] * 26 for _ in range(m + 1)]
for i in range(m - 1, -1, -1):
    nxt[i] = nxt[i + 1][:]
    nxt[i][ord(source[i]) - 97] = i
print("nxt[i] cho (a, b, c):")
for i in range(m + 1):
    print(f"  i={i}: a->{nxt[i][0]:2} b->{nxt[i][1]:2} c->{nxt[i][2]:2}")
pieces, pos = 1, 0
for ch in target:
    c = ord(ch) - 97
    note = ""
    if nxt[pos][c] == -1:
        pieces += 1
        pos = 0
        note = " (hết source: dãy con mới)"
    pos = nxt[pos][c] + 1
    print(f"ký tự {ch!r}: khớp tại {pos - 1}, pos mới = {pos}, số dãy con = {pieces}{note}")
assert pieces == 2 == shortest_bisect(source, target) == shortest_two_pointers(source, target)

nxt[i] cho (a, b, c):
  i=0: a-> 0 b-> 1 c-> 2
  i=1: a-> 3 b-> 1 c-> 2
  i=2: a-> 3 b-> 4 c-> 2
  i=3: a-> 3 b-> 4 c->-1
  i=4: a->-1 b-> 4 c->-1
  i=5: a->-1 b->-1 c->-1
ký tự 'a': khớp tại 0, pos mới = 1, số dãy con = 1
ký tự 'b': khớp tại 1, pos mới = 2, số dãy con = 1
ký tự 'a': khớp tại 3, pos mới = 4, số dãy con = 1
ký tự 'b': khớp tại 4, pos mới = 5, số dãy con = 1
ký tự 'c': khớp tại 2, pos mới = 3, số dãy con = 2 (hết source: dãy con mới)
ký tự 'a': khớp tại 3, pos mới = 4, số dãy con = 2
ký tự 'b': khớp tại 4, pos mới = 5, số dãy con = 2


### 8.3. Vì sao `reach` đơn điệu (minh họa cho tham lam)

Ta tính `reach(p)` theo định nghĩa (thử mọi `q`) và kiểm tra `reach(p) >= reach(p')` khi `p >= p'` (với `reach(p) >= p`).

In [18]:
def reach(source, target, p):
    best = p
    for q in range(p, len(target) + 1):
        if is_subsequence(target[p:q], source):
            best = q
    return best

rng = random.Random(83)
for _ in range(300):
    source = "".join(rng.choice("abc") for _ in range(rng.randint(1, 5)))
    target = "".join(rng.choice("abc") for _ in range(rng.randint(1, 9)))
    r = [reach(source, target, p) for p in range(len(target) + 1)]
    for p in range(len(target) + 1):
        for p2 in range(p + 1):
            assert r[p] >= p, (source, target)
            if r[p2] >= p:
                assert r[p] >= r[p2], (source, target, p, p2)
print("reach đơn điệu trên 300 cặp ngẫu nhiên")

reach đơn điệu trên 300 cặp ngẫu nhiên


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (hai con trỏ tham lam): ngắn nhất, dễ chứng minh, `O(m * n) <= 10^6` thao tác với ràng buộc của đề.

```python
class Solution:
    def shortestWay(self, source: str, target: str) -> int:
        if set(target) - set(source):                  # có ký tự không tồn tại trong source
            return -1
        m, n = len(source), len(target)
        pieces = 0                                     # số dãy con đã dùng
        j = 0                                          # vị trí cần khớp trong target
        while j < n:
            i = 0
            while i < m and j < n:                     # một lượt quét source = một dãy con
                if source[i] == target[j]:
                    j += 1
                i += 1
            pieces += 1
        return pieces
```

In [19]:
class Solution:
    def shortestWay(self, source: str, target: str) -> int:
        if set(target) - set(source):                  # có ký tự không tồn tại trong source
            return -1
        m, n = len(source), len(target)
        pieces = 0                                     # số dãy con đã dùng
        j = 0                                          # vị trí cần khớp trong target
        while j < n:
            i = 0
            while i < m and j < n:                     # một lượt quét source = một dãy con
                if source[i] == target[j]:
                    j += 1
                i += 1
            pieces += 1
        return pieces

sol = lambda s, t: Solution().shortestWay(s, t)
for source, target, expected in examples:
    got = sol(source, target)
    print(got, "OK" if got == expected else "SAI")
    assert got == expected

2 OK
-1 OK
3 OK


## 10. Vì sao đúng

**Điều kiện `-1`.** Nếu `target` có ký tự `c` không có trong `source` thì mọi dãy con của `source` đều không chứa `c`, nên không thể ⇒ `-1`. Ngược lại mỗi ký tự của `target` đều có trong `source`, nên mỗi lượt quét khớp được ít nhất ký tự hiện tại của `target`; do đó thuật toán tiến lên và kết thúc sau tối đa `len(target)` lượt.

**Tính tối ưu của tham lam.** Với vị trí `p` trong `target` (`0 <= p <= n`), gọi `reach(p)` là `q` lớn nhất sao cho `target[p:q]` là dãy con của `source` (`reach(p) >= p`). Hai điều:

1. Một lượt quét của thuật toán bắt đầu từ `p` kết thúc đúng tại `reach(p)`: hai con trỏ khớp ký tự `target[j]` ngay lần đầu thấy trong `source` kể từ vị trí hiện tại, đây là khớp **trái nhất**, nên khớp được nhiều ký tự liên tiếp nhất.
2. **Đơn điệu:** nếu `p >= p'` thì `reach(p) >= reach(p')`. Thật vậy, nếu `reach(p') < p` thì hiển nhiên `reach(p) >= p > reach(p')`; còn nếu `reach(p') >= p` thì `target[p:reach(p')]` là một **đoạn cuối** của `target[p':reach(p')]`, vốn là dãy con của `source`, nên nó cũng là dãy con của `source`, do đó `reach(p) >= reach(p')`.

Gọi `g_k` là vị trí tham lam đạt được sau `k` đoạn (`g_0 = 0`, `g_{k+1} = reach(g_k)`). Xét một cách chia bất kỳ thành các đoạn kết thúc tại `0 = x_0 < x_1 < x_2 < ...`. Quy nạp `x_k <= g_k`: `x_0 = g_0 = 0`; nếu `x_k <= g_k` thì đoạn `target[x_k:x_{k+1}]` là dãy con của `source` nên `x_{k+1} <= reach(x_k) <= reach(g_k) = g_{k+1}` (bất đẳng thức giữa dùng tính đơn điệu). Do đó nếu một cách chia nào đó dùng `k` đoạn để phủ hết `target` (`x_k = n`) thì `g_k = n` ngay sau `k` đoạn, tức tham lam không dùng nhiều đoạn hơn: số đoạn của tham lam **nhỏ nhất**.

**Cách 2 (DP)** dùng đúng định nghĩa: mọi cách chia hợp lệ có đoạn cuối `target[i:j]` là dãy con của `source` và phần còn lại là cách chia của `target[:i]`. **Cách 4 và 5** chỉ là cách cài đặt nhanh hơn của cùng quy trình quét: `nxt[pos][c]` và `bisect_left(idx[c], pos)` đều trả về lần xuất hiện **đầu tiên** của `c` tại hoặc sau `pos`.

**Kiểm chứng bằng máy.** Oracle độc lập (Cách 1: thử mọi cách cắt) được đối chiếu với DP trên các cặp nhỏ; sau đó DP làm oracle cho **mọi** cặp (`source` độ dài `1..4`, `target` độ dài `1..5`, bảng chữ `{a, b, c}`).

In [20]:
# (1) Cách 1 (mọi cách cắt) khớp DP trên mọi cặp rất nhỏ
cnt = 0
for ls in range(1, 4):
    for lt in range(1, 6):
        for s in product("abc", repeat=ls):
            for t in product("abc", repeat=lt):
                s_, t_ = "".join(s), "".join(t)
                assert shortest_cuts(s_, t_) == shortest_dp(s_, t_), (s_, t_)
                cnt += 1
print("oracle mọi cách cắt khớp DP trên", cnt, "cặp (source <= 3, target <= 5)")

oracle mọi cách cắt khớp DP trên 14157 cặp (source <= 3, target <= 5)


In [21]:
# (2) mọi cách khớp DP trên mọi cặp: source dài 1..4, target dài 1..5, bảng chữ {a, b, c}
impls = {"hai con trỏ": shortest_two_pointers, "bảng nxt": shortest_next_table, "bisect": shortest_bisect,
         "nộp (Cách 3)": sol, "Cách 4 (nộp)": sol_table, "Cách 5 (nộp)": sol_bisect, "Cách 2 (nộp)": sol_dp}
impls_fast = {k: v for k, v in impls.items() if k != "Cách 2 (nộp)"}      # Cách 2 nộp là DP chậm, kiểm ở bộ test ngẫu nhiên
total = 0
for ls in range(1, 5):
    for lt in range(1, 6):
        for s in product("abc", repeat=ls):
            s_ = "".join(s)
            for t in product("abc", repeat=lt):
                t_ = "".join(t)
                ref = shortest_dp(s_, t_)
                for name, fn in impls_fast.items():
                    assert fn(s_, t_) == ref, (name, s_, t_, fn(s_, t_), ref)
                total += 1
print("đã đối chiếu", total, "cặp: mọi cách khớp oracle")

đã đối chiếu 43560 cặp: mọi cách khớp oracle


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mọi cách cắt | `O(2^n * n * m)` | `O(n)` |
| 2. DP theo tiền tố | `O(n^2 * m)` | `O(n)` |
| 3. Hai con trỏ | `O(m * n)` (tệ nhất) | `O(1)` |
| 4. Bảng `nxt` | `O(26 * m + n)` | `O(26 * m)` |
| 5. `bisect_left` | `O(m + n log m)` | `O(m)` |

Đo ở cỡ tối đa (`m = n = 1000`; trừ Cách 2 vì quá chậm) với các dạng: ngẫu nhiên 26 chữ, hai chữ, nhiều dãy con (đáp án lớn), một dãy con, không thể:

In [22]:
rng = random.Random(11)
m = n = 1000
cases = {
    "ngẫu nhiên 26 chữ": ("".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(m)),
                           "".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(n))),
    "hai chữ ab": ("".join(rng.choice("ab") for _ in range(m)), "".join(rng.choice("ab") for _ in range(n))),
    "source = a*999 + b, target = (ba)*500": ("a" * 999 + "b", "ba" * 500),
    "source = (ab)*500, target = (ba)*500": ("ab" * 500, "ba" * 500),
    "target là dãy con (đáp án 1)": ("abcdefghij" * 100, "acegi" * 100),
    "toàn một chữ": ("a" * 1000, "a" * 1000),
    "source 1 ký tự, target 1000": ("a", "a" * 1000),
    "không thể": ("a" * 1000, "a" * 999 + "b"),
}
for label, (s, t) in cases.items():
    ref = shortest_bisect(s, t)
    row = [f"{label:40}"]
    for name, fn in (("hai con trỏ", sol_two), ("bảng nxt", sol_table), ("bisect", sol_bisect)):
        t0 = time.perf_counter()
        got = fn(s, t)
        dt = (time.perf_counter() - t0) * 1000
        assert got == ref, (label, name)
        row.append(f"{name} {dt:7.2f} ms")
    print(" | ".join(row), "| đáp án", ref)

ngẫu nhiên 26 chữ                        | hai con trỏ    2.27 ms | bảng nxt    2.34 ms | bisect    2.40 ms | đáp án 26
hai chữ ab                               | hai con trỏ    0.21 ms | bảng nxt    0.23 ms | bisect    0.25 ms | đáp án 2
source = a*999 + b, target = (ba)*500    | hai con trỏ   43.80 ms | bảng nxt   42.89 ms | bisect   42.96 ms | đáp án 501
source = (ab)*500, target = (ba)*500     | hai con trỏ    0.08 ms | bảng nxt    0.07 ms | bisect    0.07 ms | đáp án 2
target là dãy con (đáp án 1)             | hai con trỏ    0.10 ms | bảng nxt    0.10 ms | bisect    0.11 ms | đáp án 1
toàn một chữ                             | hai con trỏ    0.13 ms | bảng nxt    0.15 ms | bisect    0.16 ms | đáp án 1
source 1 ký tự, target 1000              | hai con trỏ    0.15 ms | bảng nxt    0.16 ms | bisect    0.16 ms | đáp án 1000
không thể                                | hai con trỏ    0.02 ms | bảng nxt    0.02 ms | bisect    0.02 ms | đáp án -1


## 12. Bộ test

Gồm: ví dụ của đề, edge case có kỳ vọng viết tay (kèm lý do), phản ví dụ cho các cách sai (tìm bằng tìm kiếm so với oracle), test ngẫu nhiên có seed và các tính chất.

In [23]:
def check_all(source, target, expected=None, oracle=True):
    results = {name: fn(source, target) for name, fn in impls.items() if name != "Cách 2 (nộp)" or len(source) * len(target) <= 400}
    first = results["nộp (Cách 3)"]
    for name, got in results.items():
        assert got == first, (source[:20], target[:20], name, got, first)
    if oracle and len(target) <= 40 and len(source) <= 40:
        assert shortest_dp(source, target) == first, (source, target)
    if expected is not None:
        assert first == expected, (source, target, first, expected)
    return first

# 12.1 ví dụ đề
for s, t, e in examples:
    check_all(s, t, e)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ("a", "a", 1),                               # giống nhau, một ký tự
    ("a", "b", -1),                              # ký tự không có
    ("abc", "abc", 1),                           # target = source
    ("abc", "cba", 3),                           # ngược hoàn toàn: mỗi dãy con một ký tự
    ("abc", "acb", 2),                           # "ac" + "b"
    ("a", "aaaa", 4),                            # source một ký tự: mỗi dãy con một ký tự
    ("ab", "bbbaaa", 6),                         # source ab: b, b, b, a, a, a không ghép được hai ký tự liền
    ("aaaaa", "aaaaaaaaaa", 2),                  # target = source * 2
    ("abcde", "edcba", 5),                       # đảo ngược, năm chữ khác nhau
    ("ab", "ababab", 3),                         # target = source * 3
    ("abcab", "ababcab", 2),                     # tính tay ở bài tập 2
    ("abc", "abcabcabc", 3),                     # target = source * 3
    ("a" * 1000, "a" * 1000, 1),                 # biên độ dài: vừa đủ
    ("a", "a" * 1000, 1000),                     # biên: đáp án lớn nhất
    ("abc", "aabbcc", 4),                        # "a" + "ab" + "bc" + "c": hai ký tự giống nhau liền nhau buộc phải sang dãy con mới
]
for s, t, e in edge:
    check_all(s, t, e)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 15 ca


In [24]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm so với oracle)
def wrong_no_missing_check_capped(source, target, cap=10 ** 4):   # quên kiểm tra ký tự thiếu: vòng lặp có thể vô hạn
    steps = 0
    pieces, j = 0, 0
    while j < len(target):
        i = 0
        while i < len(source) and j < len(target):
            if source[i] == target[j]:
                j += 1
            i += 1
        pieces += 1
        steps += 1
        if steps > cap:
            return "VÔ HẠN"
    return pieces

def wrong_substring(source, target):                 # nhầm dãy con thành chuỗi con (liên tiếp)
    if set(target) - set(source):
        return -1
    pieces, j = 0, 0
    while j < len(target):
        best = 0
        for i in range(len(source)):                 # tìm chuỗi con liên tiếp dài nhất của source bắt đầu khớp target[j:]
            k = 0
            while i + k < len(source) and j + k < len(target) and source[i + k] == target[j + k]:
                k += 1
            best = max(best, k)
        j += best
        pieces += 1
    return pieces

def wrong_count_formula(source, target):             # đếm ký tự: max theo ký tự của ceil(đếm target / đếm source)
    if set(target) - set(source):
        return -1
    cs, ct = {}, {}
    for ch in source:
        cs[ch] = cs.get(ch, 0) + 1
    for ch in target:
        ct[ch] = ct.get(ch, 0) + 1
    return max(-(-ct[ch] // cs[ch]) for ch in ct)

def wrong_bisect_right(source, target):              # lệch một: bisect_right bỏ qua đúng vị trí pos
    idx = defaultdict(list)
    for k, ch in enumerate(source):
        idx[ch].append(k)
    pieces, pos = 1, 0
    for ch in target:
        if ch not in idx:
            return -1
        lst = idx[ch]
        k = bisect_right(lst, pos)
        if k == len(lst):
            pieces += 1
            k = 0
        pos = lst[k] + 1
    return pieces

def wrong_swapped(source, target):                   # đảo vai trò source / target
    return shortest_two_pointers(target, source)

def find_counter(wrong, tries=60000, seed=9, need_possible=True, alphabet="abc"):
    rng = random.Random(seed)
    for _ in range(tries):
        s = "".join(rng.choice(alphabet) for _ in range(rng.randint(1, 5)))
        t = "".join(rng.choice(alphabet) for _ in range(rng.randint(1, 7)))
        if need_possible and set(t) - set(s):
            continue
        if wrong(s, t) != shortest_dp(s, t):
            return s, t
    return None

counters = {}
for name, fn, kw in (("quên kiểm tra ký tự thiếu (vòng lặp vô hạn)", wrong_no_missing_check_capped, {"need_possible": False}),
                     ("nhầm dãy con với chuỗi con", wrong_substring, {}),
                     ("đếm ký tự rồi chia", wrong_count_formula, {}),
                     ("bisect_right thay bisect_left", wrong_bisect_right, {}),
                     ("đảo vai trò source / target", wrong_swapped, {"need_possible": False})):
    case = find_counter(fn, **kw)
    assert case is not None, name
    counters[name] = case
    s, t = case
    print(f"{name:46} source={s!r} target={t!r}: sai -> {fn(s, t)}, đúng -> {shortest_dp(s, t)}")
    check_all(s, t)
print("12.3 phản ví dụ: OK")

quên kiểm tra ký tự thiếu (vòng lặp vô hạn)    source='ca' target='bababb': sai -> VÔ HẠN, đúng -> -1
nhầm dãy con với chuỗi con                     source='cbba' target='ca': sai -> 2, đúng -> 1
đếm ký tự rồi chia                             source='acacb' target='ababa': sai -> 2, đúng -> 3
bisect_right thay bisect_left                  source='cbba' target='ca': sai -> 2, đúng -> 1
đảo vai trò source / target                    source='cbba' target='ca': sai -> -1, đúng -> 1
12.3 phản ví dụ: OK


In [25]:
# 12.4 ngẫu nhiên nhỏ: nhiều bảng chữ, so oracle DP
rng = random.Random(4055)
for _ in range(3000):
    alpha = "abcdefghijklmnopqrstuvwxyz"[:rng.choice([1, 2, 3, 5, 26])]
    s = "".join(rng.choice(alpha) for _ in range(rng.randint(1, 12)))
    t = "".join(rng.choice(alpha) for _ in range(rng.randint(1, 15)))
    check_all(s, t)
print("12.4 ngẫu nhiên nhỏ: OK")

# 12.5 ngẫu nhiên vừa (<= 40, so DP) và lớn (<= 1000, so các cách nhanh với nhau)
for _ in range(100):
    alpha = "abcdefghijklmnopqrstuvwxyz"[:rng.choice([2, 3, 4, 8])]
    s = "".join(rng.choice(alpha) for _ in range(rng.randint(5, 40)))
    t = "".join(rng.choice(alpha) for _ in range(rng.randint(5, 40)))
    check_all(s, t)
for _ in range(30):
    alpha = "abcdefghijklmnopqrstuvwxyz"[:rng.choice([1, 2, 3, 5, 26])]
    s = "".join(rng.choice(alpha) for _ in range(rng.randint(100, 1000)))
    t = "".join(rng.choice(alpha) for _ in range(rng.randint(100, 1000)))
    check_all(s, t, oracle=False)
print("12.5 ngẫu nhiên vừa và lớn: OK")

# 12.6 tính chất
for _ in range(300):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 8)))
    t = "".join(rng.choice("abc") for _ in range(rng.randint(1, 12)))
    ans = sol(s, t)
    assert ans == -1 or 1 <= ans <= len(t)                                  # 1 <= đáp án <= len(target)
    assert (ans == 1) == is_subsequence(t, s)                               # đáp án 1 <=> target là dãy con của source
    k = rng.randint(1, 5)
    assert sol(s, s * k) == k                                               # target = source * k cần đúng k dãy con
    if ans != -1:
        prefix_answers = [sol(s, t[:j]) for j in range(1, len(t) + 1)]
        assert prefix_answers == sorted(prefix_answers)                     # thêm ký tự vào cuối không làm giảm đáp án
    assert (ans == -1) == bool(set(t) - set(s))                             # -1 <=> có ký tự thiếu
print("12.6 tính chất: OK")

12.4 ngẫu nhiên nhỏ: OK


12.5 ngẫu nhiên vừa và lớn: OK
12.6 tính chất: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên kiểm tra ký tự không có trong `source` (vòng lặp vô hạn)

Nếu `target` có ký tự không có trong `source` thì lượt quét không khớp được gì, `j` đứng yên và vòng `while j < n` **không bao giờ dừng**. (Ở đây ta đặt một giới hạn số lượt để notebook không bị treo.)

In [26]:
s, t = counters["quên kiểm tra ký tự thiếu (vòng lặp vô hạn)"]
print(f"source={s!r} target={t!r}: bản không kiểm tra ->", wrong_no_missing_check_capped(s, t), "| đúng ->", sol(s, t))
assert wrong_no_missing_check_capped(s, t) == "VÔ HẠN" and sol(s, t) == -1

source='ca' target='bababb': bản không kiểm tra -> VÔ HẠN | đúng -> -1


Vì sao sai: không có đường thoát khi `j` không tiến được. **Đúng:** kiểm tra `set(target) - set(source)` ngay từ đầu (hoặc dừng nếu một lượt quét không khớp ký tự nào).

### Lỗi 2 — Nhầm dãy con với chuỗi con (liên tiếp)

In [27]:
s, t = counters["nhầm dãy con với chuỗi con"]
print(f"source={s!r} target={t!r}: sai ->", wrong_substring(s, t), "| đúng ->", sol(s, t))
assert wrong_substring(s, t) != sol(s, t)

source='cbba' target='ca': sai -> 2 | đúng -> 1


Vì sao sai: dãy con được phép **bỏ** ký tự giữa chừng, nên một lượt quét `source` có thể khớp nhiều hơn một đoạn liên tiếp. **Đúng:** hai con trỏ, `i` luôn tiến và `j` chỉ tiến khi khớp.

### Lỗi 3 — Đếm ký tự rồi chia

Đáp án không chỉ phụ thuộc số lần mỗi ký tự xuất hiện mà còn **thứ tự**.

In [28]:
s, t = counters["đếm ký tự rồi chia"]
print(f"source={s!r} target={t!r}: sai ->", wrong_count_formula(s, t), "| đúng ->", sol(s, t))
assert wrong_count_formula(s, t) != sol(s, t)

source='acacb' target='ababa': sai -> 2 | đúng -> 3


Vì sao sai: dãy con phải giữ thứ tự, nên cùng bội số ký tự mà thứ tự đảo ngược thì cần nhiều dãy con hơn. **Đúng:** mô phỏng theo thứ tự (hai con trỏ / `nxt` / `bisect`).

### Lỗi 4 — Lệch một khi dùng `bisect`: `bisect_right` thay `bisect_left`

`pos` là vị trí **đầu tiên được phép dùng** trong `source`, nên một ký tự nằm đúng tại `pos` vẫn hợp lệ: cần `bisect_left`.

In [29]:
s, t = counters["bisect_right thay bisect_left"]
print(f"source={s!r} target={t!r}: sai ->", wrong_bisect_right(s, t), "| đúng ->", sol(s, t))
assert wrong_bisect_right(s, t) != sol(s, t)

source='cbba' target='ca': sai -> 2 | đúng -> 1


Vì sao sai: `bisect_right` bỏ qua ký tự nằm ngay tại `pos`, nên bắt đầu dãy con mới quá sớm. **Đúng:** `bisect_left(lst, pos)`.

### Lỗi 5 — Đảo vai trò `source` / `target`

Ta cắt `target` thành các dãy con của `source`, không phải ngược lại.

In [30]:
s, t = counters["đảo vai trò source / target"]
print(f"source={s!r} target={t!r}: sai ->", wrong_swapped(s, t), "| đúng ->", sol(s, t))
assert wrong_swapped(s, t) != sol(s, t)

source='cbba' target='ca': sai -> -1 | đúng -> 1


Vì sao sai: hai chuỗi không đối xứng (`source` được dùng lại nhiều lần, `target` chỉ đọc một lần). **Đúng:** con trỏ `i` chạy trên `source`, con trỏ `j` chạy trên `target`, mỗi lượt quét `source` tạo một dãy con.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "chia chuỗi thành ít đoạn nhất, mỗi đoạn thuộc một lớp (dãy con của chuỗi cho trước)" ⇒ **tham lam**: mỗi đoạn lấy dài nhất, chứng minh bằng tính đơn điệu `reach`.
- Hai con trỏ trên hai chuỗi là mẫu "khớp dãy con" cơ bản (Is Subsequence, 392).
- Khi phải khớp nhiều lần, tiền xử lý **vị trí xuất hiện kế tiếp** (`nxt` hoặc `bisect`) thay vì quét lại.
- Một dạng nhanh trong Python: `pos = source.find(ch, pos)` tìm ký tự kế tiếp bằng C; `-1` nghĩa là hết, sang dãy con mới (`pos = source.find(ch) + 1`).
- Kiểm tra `-1` bằng `set(target) <= set(source)` ngay từ đầu.
- Cùng họ: Is Subsequence (392), Number of Matching Subsequences (792), Append Characters to String to Make Subsequence (2486).
- Test nên có: ký tự thiếu, target = source nhân `k`, target đảo ngược, `source` một ký tự, `n = m = 1000`.

## 15. Biến thể

1. **In ra các dãy con** (không chỉ số lượng): xem bài tập 5.
2. **Number of Matching Subsequences (792):** đếm các từ là dãy con của một chuỗi; dùng đúng danh sách vị trí + `bisect_left`.
3. **Is Subsequence (392):** trường hợp đặc biệt `pieces = 1`.
4. Bài liên quan: 392, 792, 2486.

Kiểm biến thể 2 bằng oracle kiểm tra từng từ:

In [31]:
def num_matching_subseq(s: str, words: List[str]) -> int:
    idx = defaultdict(list)
    for k, ch in enumerate(s):
        idx[ch].append(k)
    count = 0
    for w in words:
        pos, ok = 0, True
        for ch in w:
            lst = idx.get(ch)
            if not lst:
                ok = False
                break
            k = bisect_left(lst, pos)
            if k == len(lst):
                ok = False
                break
            pos = lst[k] + 1
        count += ok
    return count

rng = random.Random(15)
for _ in range(1000):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 10)))
    words = ["".join(rng.choice("abc") for _ in range(rng.randint(1, 5))) for _ in range(rng.randint(1, 8))]
    assert num_matching_subseq(s, words) == sum(is_subsequence(w, s) for w in words), (s, words)
assert num_matching_subseq("abcde", ["a", "bb", "acd", "ace"]) == 3
print("biến thể Number of Matching Subsequences: kiểm bằng oracle")

biến thể Number of Matching Subsequences: kiểm bằng oracle


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi cách cắt | Theo định nghĩa | `O(2^n ...)` | `O(n)` | Oracle cho dữ liệu cực nhỏ |
| 2. DP theo tiền tố | `dp[j] = min(dp[i] + 1)` | `O(n^2 * m)` | `O(n)` | Oracle cho dữ liệu nhỏ |
| 3. Hai con trỏ | Mỗi lượt quét = một dãy con | `O(m * n)` | `O(1)` | **Cách nộp chính** |
| 4. Bảng `nxt` | Ký tự kế tiếp `O(1)` | `O(26 m + n)` | `O(26 m)` | Khi `target` rất dài, nhiều truy vấn |
| 5. `bisect_left` | Danh sách vị trí + nhị phân | `O(m + n log m)` | `O(m)` | Gọn, bộ nhớ nhỏ hơn bảng `nxt` |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
print(Solution().shortestWay("abc", "abcbc"))
print(Solution().shortestWay("xyz", "xzyxz"))
print(Solution().shortestWay("abc", "acdbc"))
```

In [32]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None      # ("abc", "abcbc")
my_guess_2 = None      # ("xyz", "xzyxz")
my_guess_3 = None      # ("abc", "acdbc")

### Bài 2 — Tính tay (dễ-vừa)

`source = "abcab"`, `target = "ababcab"`. Liệt kê các dãy con theo tham lam và ghi đáp án.

In [33]:
# Bài 2: điền danh sách các dãy con và đáp án
hand_pieces, hand_answer = None, None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành hai con trỏ.

In [34]:
def exercise_3(source: str, target: str) -> int:
    # TODO 1: nếu target có ký tự không có trong source thì trả về -1
    pieces, j = 0, 0
    while j < len(target):
        i = 0
        while i < len(source) and j < len(target):
            # TODO 2: nếu source[i] == target[j] thì tiến j
            # TODO 3: tiến i
            pass
        pieces += 1
    return pieces


# for s, t, e in examples:
#     assert exercise_3(s, t) == e

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 5, tự viết `exercise_4(source, target)` bằng **danh sách vị trí + `bisect_left`**.

In [35]:
def exercise_4(source: str, target: str) -> int:
    # TODO
    pass


# for s, t, e in examples:
#     assert exercise_4(s, t) == e

### Bài 5 — Mini challenge: in ra các dãy con (khó)

Viết `exercise_5(source, target)` trả về **danh sách các dãy con** (các chuỗi) có số lượng ít nhất mà nối lại bằng `target`, hoặc `None` nếu không thể. Có thể có nhiều đáp án đúng nên kiểm bằng hàm kiểm tra `check_pieces`.

```python
exercise_5("abc", "abcbc")   -> ["abc", "bc"]
exercise_5("xyz", "xzyxz")   -> ["xz", "y", "xz"]
exercise_5("abc", "acdbc")   -> None
```
Gợi ý: như hai con trỏ, nhưng ghi lại `target[start:j]` ở cuối mỗi lượt quét.

In [36]:
def exercise_5(source: str, target: str) -> Optional[List[str]]:
    # TODO
    pass


def check_pieces(source: str, target: str, pieces) -> bool:
    expected = shortest_dp(source, target)
    if expected == -1:
        return pieces is None
    if pieces is None:
        return False
    return ("".join(pieces) == target and all(p and is_subsequence(p, source) for p in pieces) and len(pieces) == expected)


# for s, t, e in examples:
#     assert check_pieces(s, t, exercise_5(s, t))

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: mỗi lượt quét `source` tạo một dãy con; `-1` nếu thiếu ký tự.
- Mức 2: `"abcbc"`: quét 1 khớp `abc`, quét 2 khớp `bc`; `"xzyxz"`: `xz`, `y`, `xz`.
- Mức 3: `2`, `3`, `-1`.

**Bài 2**
- Mức 1: quét `source = "abcab"` khớp tham lam `target = "ababcab"`.
- Mức 2: lượt 1: `a`, `b`, (`c` không khớp `a`), `a`, `b` khớp: được `"abab"`; lượt 2: `c`, `a`, `b`.
- Mức 3: `["abab", "cab"]`, đáp án `2`.

**Bài 3**
- Mức 1: `if set(target) - set(source): return -1`.
- Mức 2: `if source[i] == target[j]: j += 1`.
- Mức 3: `i += 1` sau mỗi lần so sánh (luôn tiến `i`).

**Bài 4**
- Mức 1: `idx[ch]` là danh sách vị trí của `ch` trong `source`.
- Mức 2: `k = bisect_left(lst, pos)`; nếu `k == len(lst)` thì dãy con mới, `k = 0`.
- Mức 3: `pos = lst[k] + 1`; khởi tạo `pieces = 1`.

**Bài 5**
- Mức 1: kiểm tra ký tự thiếu trước, trả `None`.
- Mức 2: mỗi lượt quét lưu `start = j` trước và `target[start:j]` sau.
- Mức 3: thêm chuỗi đó vào danh sách `pieces` rồi lặp tiếp.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [37]:
r1, r2, r3 = Solution().shortestWay("abc", "abcbc"), Solution().shortestWay("xyz", "xzyxz"), Solution().shortestWay("abc", "acdbc")
print(r1, r2, r3)
assert (r1, r2, r3) == (2, 3, -1)

2 3 -1


### Đáp án Bài 2

Lượt 1: `a`(0) khớp `a`, `b`(1) khớp `b`, `c`(2) không khớp `a`, `a`(3) khớp `a`, `b`(4) khớp `b`: được `"abab"` (j = 4). Lượt 2: `a` không khớp `c`, `b` không khớp `c`, `c`(2) khớp `c`, `a`(3) khớp `a`, `b`(4) khớp `b`: được `"cab"`. Đáp án `2`.

In [38]:
hand_pieces_solution, hand_answer_solution = ["abab", "cab"], 2
assert "".join(hand_pieces_solution) == "ababcab"
assert all(is_subsequence(p, "abcab") for p in hand_pieces_solution)
assert len(hand_pieces_solution) == hand_answer_solution == Solution().shortestWay("abcab", "ababcab") == shortest_dp("abcab", "ababcab")
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [39]:
def exercise_3_solution(source: str, target: str) -> int:
    if set(target) - set(source):
        return -1
    pieces, j = 0, 0
    while j < len(target):
        i = 0
        while i < len(source) and j < len(target):
            if source[i] == target[j]:
                j += 1
            i += 1
        pieces += 1
    return pieces

for s, t, e in examples:
    assert exercise_3_solution(s, t) == e
rng = random.Random(3)
for _ in range(500):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 6)))
    t = "".join(rng.choice("abcd") for _ in range(rng.randint(1, 8)))
    assert exercise_3_solution(s, t) == shortest_dp(s, t), (s, t)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [40]:
def exercise_4_solution(source: str, target: str) -> int:
    idx = defaultdict(list)
    for k, ch in enumerate(source):
        idx[ch].append(k)
    pieces, pos = 1, 0
    for ch in target:
        if ch not in idx:
            return -1
        lst = idx[ch]
        k = bisect_left(lst, pos)
        if k == len(lst):
            pieces += 1
            k = 0
        pos = lst[k] + 1
    return pieces

for s, t, e in examples:
    assert exercise_4_solution(s, t) == e
rng = random.Random(4)
for _ in range(500):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 6)))
    t = "".join(rng.choice("abcd") for _ in range(rng.randint(1, 8)))
    assert exercise_4_solution(s, t) == shortest_dp(s, t), (s, t)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [41]:
def exercise_5_solution(source: str, target: str) -> Optional[List[str]]:
    if set(target) - set(source):
        return None
    pieces, j = [], 0
    while j < len(target):
        start, i = j, 0
        while i < len(source) and j < len(target):
            if source[i] == target[j]:
                j += 1
            i += 1
        pieces.append(target[start:j])
    return pieces

def check_pieces_solution(source, target, pieces):
    expected = shortest_dp(source, target)
    if expected == -1:
        return pieces is None
    return (pieces is not None and "".join(pieces) == target
            and all(p and is_subsequence(p, source) for p in pieces) and len(pieces) == expected)

for s, t, e in examples:
    assert check_pieces_solution(s, t, exercise_5_solution(s, t))
assert exercise_5_solution("abc", "abcbc") == ["abc", "bc"]
assert exercise_5_solution("xyz", "xzyxz") == ["xz", "y", "xz"]
assert exercise_5_solution("abc", "acdbc") is None
rng = random.Random(5)
for _ in range(500):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 6)))
    t = "".join(rng.choice("abcd") for _ in range(rng.randint(1, 8)))
    assert check_pieces_solution(s, t, exercise_5_solution(s, t)), (s, t)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (hai con trỏ tham lam)

```python
class Solution:
    def shortestWay(self, source: str, target: str) -> int:
        if set(target) - set(source):                  # ký tự thiếu -> không thể
            return -1
        m, n = len(source), len(target)
        pieces, j = 0, 0
        while j < n:
            i = 0
            while i < m and j < n:                     # một lượt quét source = một dãy con
                if source[i] == target[j]:
                    j += 1
                i += 1
            pieces += 1
        return pieces
```

Công thức nhớ nhanh: **mỗi lượt quét `source` khớp càng nhiều ký tự liên tiếp của `target` càng tốt; hết `source` thì tính một dãy con và quét lại từ đầu**.

### Checklist trước khi submit

- [ ] Đã kiểm tra ký tự của `target` không có trong `source` (trả `-1`, tránh vòng lặp vô hạn)?
- [ ] `i` luôn tiến, `j` chỉ tiến khi khớp?
- [ ] Mỗi lượt quét `source` cộng đúng `1` vào kết quả (kể cả lượt cuối)?
- [ ] Nếu dùng `bisect`: dùng `bisect_left(lst, pos)` và `pos = lst[k] + 1`?
- [ ] Đã phân biệt dãy con (không liên tiếp) với chuỗi con?
- [ ] Đã thử: target là dãy con (đáp án `1`), ký tự thiếu, `source` một ký tự, target ngược, `m = n = 1000`?